# AsyncGeoTIFFReader — async COG reads with `asyncio.gather`

`AsyncGeoTIFFReader` is georeader's async-native COG reader. It satisfies the
`AsyncGeoData` protocol, exposes the same metadata surface as `RasterioReader`,
and lets you fan out hundreds of reads concurrently from a single process with
`asyncio.gather`. It is a **thin adapter** over
[`developmentseed/async-geotiff`](https://github.com/developmentseed/async-geotiff):
IFD walk, tile-fetch math, decompression dispatch, and request coalescing all
live upstream — we contribute the carrier translation, the lazy-view pattern,
and protocol conformance with the rest of georeader.

**Audience.** Anyone who has used `RasterioReader` and is wondering when to
reach for the async sibling — what changes, what stays the same, what the
two-phase laziness model looks like, which `read.*` module functions work
out of the box, and how to do the things `async-geotiff` deliberately
doesn't (warp, resample).

This notebook runs against a small local fixture — no credentials, no
network. The patterns translate to S3 / GCS / Azure by swapping `LocalStore`
for the appropriate `obstore.store.*` class (last section shows the
pseudocode).


## A 60-second async/await primer

If you've never written async Python before, here is the minimum vocabulary
to follow the rest of this notebook.

- A function declared with `async def` is a **coroutine**. Calling it does
  **not** run the body — it returns a coroutine object that has to be
  scheduled on an **event loop** to execute.
- The `await` keyword runs a coroutine and gives back its return value.
  You can only `await` inside an `async def` (or at the top level of a
  Jupyter cell, which Jupyter wraps in one for you).
- `asyncio.gather(coro1, coro2, ...)` schedules many coroutines on the
  event loop **at the same time**. While one is parked on I/O, the others
  can make progress. The result is the list of return values, in order.
- `asyncio.run(main())` is how you bootstrap an event loop from a normal
  sync entry point (a script's `if __name__ == "__main__":`, a CLI, etc).
  In Jupyter you don't need this — the cell already has a running loop.

Concurrent vs sequential, by example::

    # Sequential — each await blocks the loop until the read returns
    gt1 = await reader.read_from_window(w1).load()
    gt2 = await reader.read_from_window(w2).load()
    gt3 = await reader.read_from_window(w3).load()
    # Total wall time ≈ sum of individual latencies.

    # Concurrent — gather schedules all three, the event loop multiplexes
    gt1, gt2, gt3 = await asyncio.gather(
        reader.read_from_window(w1).load(),
        reader.read_from_window(w2).load(),
        reader.read_from_window(w3).load(),
    )
    # Total wall time ≈ max of individual latencies (when bytes-in-flight
    # is the bottleneck, which is the cloud-read case).

That's it for this notebook. The
[Python asyncio docs](https://docs.python.org/3/library/asyncio-task.html)
go deeper on tasks, queues, cancellation, and the rest.


## When is async actually worth it?

Async is **only** a win when your reads are bottlenecked on
**concurrent I/O latency** — typically cloud reads where each
`read_from_window` is a 50–200 ms HTTP round-trip and you have many of
them to issue. Then `asyncio.gather` overlaps the wait times and your
wall-clock cost is close to the slowest single read instead of the sum.

Async is **not** a win when:

- Reads are local (memory-mapped or fast SSD). Per-call latency is too
  small for overlap to matter; the event-loop overhead can even dominate.
- You issue one read at a time. No concurrency, no overlap, no win.
- You need per-row CPU-heavy work between reads. Async doesn't parallelise
  CPU — it only overlaps waiting. Use `multiprocessing` for CPU.

If you're not sure, profile your sync path first. Most of the time
`RasterioReader` is the right answer.


## How `RasterioReader` fetches bytes (sync, GDAL-backed)

For context, the sync sibling routes bytes through one of three paths,
controlled by the `opener=` / `fs=` constructor knobs (see
[`bytes_path_knobs.ipynb`](bytes_path_knobs.ipynb)):

<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; max-width: 760px; margin: 1em 0;">
  <div style="border: 2px solid #1976d2; border-radius: 10px; padding: 14px; background: #e3f2fd;">
    <strong>Your code</strong><br/>
    <code>reader.load()&nbsp;/&nbsp;reader.read_from_window(w)</code>
  </div>
  <div style="text-align: center; font-size: 1.4em; color: #555; line-height: 1;">&darr;</div>
  <div style="border: 2px solid #f57c00; border-radius: 10px; padding: 14px; background: #fff3e0;">
    <strong>RasterioReader</strong> &middot; wraps <code>rasterio.open(...)</code>
  </div>
  <div style="text-align: center; font-size: 1.4em; color: #555; line-height: 1;">&darr;</div>
  <table style="width: 100%; border-collapse: separate; border-spacing: 10px 0; margin: 0;">
    <tr>
      <td style="border: 2px solid #7b1fa2; border-radius: 10px; padding: 10px; background: #f3e5f5; text-align: center; vertical-align: top; width: 33%;">
        <strong>GDAL VSI</strong><br/>
        <em style="color: #555;">(default)</em><br/>
        <code>opener=None, fs=None</code><br/>
        <small style="color: #555;">libcurl in C; fastest cloud path</small>
      </td>
      <td style="border: 2px solid #7b1fa2; border-radius: 10px; padding: 10px; background: #f3e5f5; text-align: center; vertical-align: top; width: 34%;">
        <strong>Python opener</strong><br/>
        <em style="color: #555;">(custom callback)</em><br/>
        <code>opener=callable</code><br/>
        <small style="color: #555;">custom HTTP / auth, sync facade over async</small>
      </td>
      <td style="border: 2px solid #7b1fa2; border-radius: 10px; padding: 10px; background: #f3e5f5; text-align: center; vertical-align: top; width: 33%;">
        <strong>fsspec</strong><br/>
        <em style="color: #555;">(shortcut for opener)</em><br/>
        <code>fs=fsspec_fs</code><br/>
        <small style="color: #555;">niche backends, custom auth</small>
      </td>
    </tr>
  </table>
  <div style="text-align: center; font-size: 1.4em; color: #555; line-height: 1;">&darr;</div>
  <div style="border: 2px solid #388e3c; border-radius: 10px; padding: 14px; background: #c8e6c9; text-align: center;">
    <strong>Cloud storage / local disk</strong><br/>
    <small style="color: #555;">S3, GCS, Azure, HTTPS, FTP, SFTP, GitHub, local</small>
  </div>
</div>

All three paths are **synchronous**. The reader opens a fresh `rasterio.DatasetReader`
on each `read()` call, which keeps it pickleable across `multiprocessing` /
`joblib` / Dask workers.


## How `AsyncGeoTIFFReader` fetches bytes (async, GDAL-free)

<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; max-width: 760px; margin: 1em 0;">
  <div style="border: 2px solid #1976d2; border-radius: 10px; padding: 14px; background: #e3f2fd;">
    <strong>Your code</strong><br/>
    <code>await&nbsp;reader.read_from_window(w)</code>
  </div>
  <div style="text-align: center; font-size: 1.4em; color: #555; line-height: 1;">&darr; delegates to</div>
  <div style="border: 2px solid #f57c00; border-radius: 10px; padding: 14px; background: #fff3e0;">
    <strong>AsyncGeoTIFFReader</strong> &middot; <em style="color:#555;">this package, thin adapter, lazy-view pattern</em><br/>
    <small>Carrier translation (RasterArray &rarr; GeoTensor), protocol conformance, overview indexing</small>
  </div>
  <div style="text-align: center; font-size: 1.4em; color: #555; line-height: 1;">&darr; wraps</div>
  <div style="border: 2px solid #c2185b; border-radius: 10px; padding: 14px; background: #fce4ec;">
    <strong>async_geotiff.GeoTIFF</strong> &middot; <em style="color:#555;">DevSeed</em><br/>
    <small>IFD walk, GeoKey parsing, GDAL-metadata parsing, tile-fetch math, RasterArray assembly</small>
  </div>
  <div style="text-align: center; font-size: 1.4em; color: #555; line-height: 1;">&darr; (Rust core, off the event loop)</div>
  <div style="border: 2px solid #c2185b; border-radius: 10px; padding: 14px; background: #fce4ec;">
    <strong>async-tiff</strong><br/>
    <small>Per-tile range fetch &middot; decompression &middot; <strong>request coalescing for adjacent tiles</strong></small>
  </div>
  <div style="text-align: center; font-size: 1.4em; color: #555; line-height: 1;">&darr; via</div>
  <div style="border: 2px solid #c2185b; border-radius: 10px; padding: 14px; background: #fce4ec;">
    <strong>obspec.AsyncStore</strong><br/>
    <small><code>obstore.store</code>: S3Store &middot; GCSStore &middot; AzureStore &middot; HTTPStore &middot; LocalStore</small>
  </div>
  <div style="text-align: center; font-size: 1.4em; color: #555; line-height: 1;">&darr;</div>
  <div style="border: 2px solid #388e3c; border-radius: 10px; padding: 14px; background: #c8e6c9; text-align: center;">
    <strong>Cloud storage / local disk</strong>
  </div>
</div>

Boxes below `AsyncGeoTIFFReader` are all external dependencies. The Rust core
coalesces adjacent tile reads within one `await` call, so a single window read
already issues parallel HTTP range requests under the hood. The header is
fetched once on `open()`; pixel bytes are fetched per `read_*` / `load` call.


## Which reader should I use?

| Scenario | Reader | Why |
|---|---|---|
| Notebook exploration, batch scripts, single scenes | `RasterioReader` | Sync API is simpler; one read at a time is the common case. |
| JP2, NetCDF, HDF5, GRIB, non-COG formats | `RasterioReader` | Async reader is TIFF/COG only. |
| Cross-CRS reads via WarpedVRT | `RasterioReader` | Async reader does not warp (see mini-solution below). |
| `multiprocessing` / `joblib` / Dask workers | `RasterioReader` | Opens fresh per `read()`, pickleable across processes. |
| Tile servers fanning out 100s of concurrent reads | `AsyncGeoTIFFReader` | `asyncio.gather` shines when reads are network-bound. |
| Async ML inference services that read many chips per request | `AsyncGeoTIFFReader` | Concurrent fetch from one process, one event loop. |
| Cloud-heavy COG workflows that want to skip GDAL | `AsyncGeoTIFFReader` | `obstore` is Rust + HTTP/2 + native parallel ranges. |

If you're not sure, **start with `RasterioReader`**. Switch when profiling
shows you're bottlenecked on concurrent cloud reads from one process.


## Setup — build a small local COG fixture

We use `georeader.save.save_cog` to build a tiled COG with overviews in
two lines. Behind the scenes, `save_cog` writes the GeoTensor, opens it
in update mode, builds an overview ladder sized to the raster, and emits
a COG-driver TIFF. `BLOCKSIZE=64` picks a small tile size so a 256×256
raster gets two overview levels (`[2, 4]`); the default 256-tile size
would give zero overviews on a raster this small.


In [1]:
import os
import tempfile

import numpy as np
from rasterio.transform import from_origin

from georeader.geotensor import GeoTensor
from georeader.save import save_cog

tmpdir = tempfile.mkdtemp()
fname = "demo.tif"
fixture_path = os.path.join(tmpdir, fname)

np.random.seed(0)
data = np.random.randint(0, 5000, size=(3, 256, 256), dtype=np.int16)

# Build a GeoTensor and save it as a COG via save_cog. It handles tiling,
# overview-ladder sizing, and the COG driver knobs.
gt_fixture = GeoTensor(
    values=data,
    transform=from_origin(500000.0, 4600000.0, 10.0, 10.0),
    crs="EPSG:32631",
    fill_value_default=0,
)
save_cog(gt_fixture, fixture_path, profile={"nodata": 0, "BLOCKSIZE": 64})

print(f"Fixture: {fixture_path}")


Fixture: /tmp/tmp5be9dpt6/demo.tif


## Opening a reader — two-phase laziness

Construction is **two phases**:

1. **`AsyncGeoTIFFReader(path, store=...)`** — pure constructor, no I/O.
   Property accessors raise `RuntimeError` before `open()`.
2. **`await AsyncGeoTIFFReader.open(...)`** — fetches only the COG header
   (the IFD chain). Cheap. After this, sync metadata properties are instant.

Pixel bytes are fetched on the *first* `await reader.read_*(...)` /
`await reader.load()` call.


In [2]:
from obstore.store import LocalStore
from georeader.async_geotiff_reader import AsyncGeoTIFFReader

store = LocalStore(prefix=tmpdir)

# Phase 1: construct (no I/O). Properties raise until open() is awaited.
reader = AsyncGeoTIFFReader(fname, store=store)
print(f"Before open: {reader}")

try:
    _ = reader.crs
except RuntimeError as e:
    print(f"  reader.crs raised: {e}")


Before open: AsyncGeoTIFFReader(path_or_url='demo.tif', overview_level=None, unopened)
  reader.crs raised: AsyncGeoTIFFReader not opened — call `await AsyncGeoTIFFReader.open(...)` or use `async with`.


In [3]:
# Phase 2: open() fetches the IFD chain only. Pixels are NOT yet downloaded.
reader = await AsyncGeoTIFFReader.open(fname, store=store)
print(f"After open:  {reader}")


After open:  
         path_or_url:        demo.tif
         overview_level:     None
         Shape:              (3, 256, 256)
         Resolution:         (10.0, 10.0)
         Bounds:             (500000.0, 4597440.0, 502560.0, 4600000.0)
         CRS:                EPSG:32631
         fill_value_default: 0.0
         Transform:          | 10.00, 0.00, 500000.00|
                             | 0.00,-10.00, 4600000.00|
                             | 0.00, 0.00, 1.00|



## Sync metadata properties (after `open()`)

Same surface as `RasterioReader`: `crs`, `transform`, `shape`, `width`,
`height`, `bounds`, `res`, `dtype`, `fill_value_default`, `dims`, plus the
`footprint(crs)` method. All sync, all instant — they just read fields off
the already-fetched header.

In [4]:
# `__repr__` shows every field in one go (same layout as `RasterioReader`).
print(reader)

# All fields are also accessible programmatically — useful when you need
# to pass them to other code rather than just look at them.
assert reader.shape == (3, 256, 256)
assert reader.dtype == np.int16
assert str(reader.crs) == "EPSG:32631"
assert reader.dims == ["band", "y", "x"]
assert reader.fill_value_default == 0
print("\n(individual property access also works — assertions above passed)")



         path_or_url:        demo.tif
         overview_level:     None
         Shape:              (3, 256, 256)
         Resolution:         (10.0, 10.0)
         Bounds:             (500000.0, 4597440.0, 502560.0, 4600000.0)
         CRS:                EPSG:32631
         fill_value_default: 0.0
         Transform:          | 10.00, 0.00, 500000.00|
                             | 0.00,-10.00, 4600000.00|
                             | 0.00, 0.00, 1.00|


(individual property access also works — assertions above passed)


## Reading data — the view + load pattern

`AsyncGeoTIFFReader` mirrors `RasterioReader`'s laziness pattern:

- **`reader.read_from_window(window)`** is **sync** — it returns a new
  `AsyncGeoTIFFReader` with a `window_focus` set, no I/O. You can chain it,
  inspect `.shape` / `.bounds` / `.transform`, and decide later whether to
  materialise.
- **`await view.load()`** is **async** — this is where the bytes actually
  travel. Returns a `GeoTensor` with `.values`, `.transform`, `.crs`, etc.

This split is what makes the `read.*` module functions work polymorphically
with both readers (more on that below).


In [5]:
import rasterio.windows
from georeader.rasterio_reader import RasterioReader

win = rasterio.windows.Window(col_off=32, row_off=16, width=64, height=48)

# AsyncGeoTIFFReader: sync view → async load
async_view = reader.read_from_window(win)            # sync, no I/O
print(f"Async view (no I/O yet): shape={async_view.shape}, type={type(async_view).__name__}")
async_gt = await async_view.load()                    # async fetch

# RasterioReader has the same shape: sync view → sync load
sync_view = RasterioReader(fixture_path).read_from_window(win)
sync_gt = sync_view.load()

print(f"async_gt.values.shape: {async_gt.values.shape}")
print(f"sync_gt.values.shape:  {sync_gt.values.shape}")
print(f"Numerically identical: {np.array_equal(async_gt.values, sync_gt.values)}")


Async view (no I/O yet): shape=(3, 48, 64), type=AsyncGeoTIFFReader
async_gt.values.shape: (3, 48, 64)
sync_gt.values.shape:  (3, 48, 64)
Numerically identical: True


### Quick reference

| What you write | What returns | Sync or async? | When the bytes travel |
|---|---|---|---|
| `reader` | `AsyncGeoTIFFReader` | sync | After `await open()`, only the header |
| `reader.read_from_window(w)` | `AsyncGeoTIFFReader` (windowed view) | sync | Never — just metadata math |
| `await view.load()` | `GeoTensor` | async | Here |
| `await reader.load()` | `GeoTensor` (whole raster) | async | Here |
| `view.shape` / `view.bounds` / `view.transform` | tuple / Affine | sync | Never |


## Overviews — reading at lower resolutions

COGs commonly ship with a pyramid of pre-downsampled overviews. Each overview
is a smaller copy of the full raster, useful for quick previews, tile-server
rendering at low zoom, or saving bandwidth when you don't need full detail.

Two helpers expose the overview chain:

- **`reader.overviews()`** — returns the decimation factors, e.g. `[2, 4, 8]`.
  Same convention as `RasterioReader.overviews()`.
- **`reader.reader_overview(level)`** — returns a new reader pinned to the
  i-th overview (0-based). Negative indexing supported (`-1` is the
  coarsest overview, *not* full res — matches the parent's convention).

You can also pass `overview_level=` directly to `AsyncGeoTIFFReader.open(...)`.


In [6]:
# Inspect overviews via the public API (no reaching into _geotiff internals).
print(f"Overview decimation factors: {reader.overviews()}")

# Three readers at three resolutions.
reader_full = await AsyncGeoTIFFReader.open(fname, store=store)               # primary IFD
reader_ovr0 = await AsyncGeoTIFFReader.open(fname, store=store, overview_level=0)
reader_ovr1 = await AsyncGeoTIFFReader.open(fname, store=store, overview_level=1)

print()
print(f"Full resolution  : shape={reader_full.shape}, res={reader_full.res}")
print(f"Overview 0 (2x)  : shape={reader_ovr0.shape}, res={reader_ovr0.res}")
print(f"Overview 1 (4x)  : shape={reader_ovr1.shape}, res={reader_ovr1.res}")


Overview decimation factors: [2, 4]

Full resolution  : shape=(3, 256, 256), res=(10.0, 10.0)
Overview 0 (2x)  : shape=(3, 128, 128), res=(20.0, 20.0)
Overview 1 (4x)  : shape=(3, 64, 64), res=(40.0, 40.0)


In [7]:
# Loading the whole raster at each level pays bytes proportional to grid size
gt_full = await reader_full.load()
gt_ovr0 = await reader_ovr0.load()
gt_ovr1 = await reader_ovr1.load()

print(f"Full-res load: shape={gt_full.values.shape}, bytes={gt_full.values.nbytes:>8,}")
print(f"Overview 0   : shape={gt_ovr0.values.shape}, bytes={gt_ovr0.values.nbytes:>8,}  ({gt_full.values.nbytes/gt_ovr0.values.nbytes:.1f}x smaller)")
print(f"Overview 1   : shape={gt_ovr1.values.shape}, bytes={gt_ovr1.values.nbytes:>8,}  ({gt_full.values.nbytes/gt_ovr1.values.nbytes:.1f}x smaller)")


Full-res load: shape=(3, 256, 256), bytes= 393,216
Overview 0   : shape=(3, 128, 128), bytes=  98,304  (4.0x smaller)
Overview 1   : shape=(3, 64, 64), bytes=  24,576  (16.0x smaller)


## Concurrent fan-out — the killer feature

The point of going async is to fan out many reads concurrently from one
process. Build the views synchronously (instant), then `asyncio.gather` the
loads.

**Honest disclaimer about local fixtures:** speedups from async only show up
against meaningful per-read latency — typically *cloud* reads. Against a
local file, the overhead can even dominate. Don't draw timing conclusions
from this cell; it proves the fan-out *works*, which is the actual question.
Real wins arrive when each `read_from_window().load()` is a 50–200 ms network
round-trip and you have 100+ of them to issue.


In [8]:
import asyncio

# 16 non-overlapping 64x64 windows tiling the 256x256 raster
windows = [
    rasterio.windows.Window(col_off=c, row_off=r, width=64, height=64)
    for r in range(0, 256, 64) for c in range(0, 256, 64)
]

# Build the views synchronously (instant), gather the loads concurrently.
results = await asyncio.gather(
    *[reader.read_from_window(w).load() for w in windows]
)

print(f"Issued {len(windows)} concurrent loads against one reader")
print(f"All shapes correct: {all(r.values.shape == (3, 64, 64) for r in results)}")


Issued 16 concurrent loads against one reader
All shapes correct: True


## Tile-aligned fan-out with `block_windows()`

The COG's internal tile grid is the natural fan-out unit. Reading
non-aligned windows forces `async-geotiff` to fetch the tiles that *cover*
your window and crop afterward — wasting bytes over the wire. Reading
tile-aligned windows fetches exactly the tiles you need, nothing more.

`reader.block_windows()` returns the internal tile grid as
`[((row_idx, col_idx), Window), ...]`. Same signature as
`RasterioReader.block_windows()`.


In [9]:
# The COG's native tile grid — fan out aligned with this for best efficiency.
blocks = reader.block_windows()
print(f"COG has {len(blocks)} tiles of {blocks[0][1].width}x{blocks[0][1].height}")

# Tile-aligned fan-out
tile_chips = await asyncio.gather(
    *[reader.read_from_window(w).load() for _, w in blocks]
)
print(f"Read {len(tile_chips)} tile-aligned chips covering the full raster")
print(f"Total pixels: {sum(c.values.shape[1] * c.values.shape[2] for c in tile_chips)}")


COG has 16 tiles of 64x64


Read 16 tile-aligned chips covering the full raster
Total pixels: 65536


## `async with` — context manager

When you don't want to manage `open()` / `aclose()` yourself, use the async
context manager. It opens lazily on enter and runs `aclose()` on exit
(currently a no-op since `obstore` pools its own connections).

In [10]:
ctx_reader = AsyncGeoTIFFReader(fname, store=store)

async with ctx_reader:
    gt = await ctx_reader.load()
    print(f"Inside the context: shape={gt.values.shape}, dtype={gt.values.dtype}")


Inside the context: shape=(3, 256, 256), dtype=int16


## Reading via the `asyncread` module

`georeader.read.*` is the **sync** read/reproject family for `RasterioReader`
and other `GeoData` inputs. `georeader.asyncread.*` is the **async** sibling
with the same nine functions, the same signatures, and the same semantics —
just declared `async def` and `await`-able. Use it for `AsyncGeoTIFFReader`
(or any other `AsyncGeoData` input).

### Side-by-side

| Sync (`read.*`) on `RasterioReader` | Async (`asyncread.*`) on `AsyncGeoTIFFReader` |
|---|---|
| `read.read_from_window(r, w, trigger_load=True)` | `await asyncread.read_from_window(r, w, trigger_load=True)` |
| `read.read_from_bounds(r, bounds)` | `await asyncread.read_from_bounds(r, bounds, trigger_load=True)` |
| `read.read_from_polygon(r, poly)` | `await asyncread.read_from_polygon(r, poly, trigger_load=True)` |
| `read.read_from_center_coords(r, c, shape)` | `await asyncread.read_from_center_coords(r, c, shape, trigger_load=True)` |
| `read.read_reproject(r, ...)` | `await asyncread.read_reproject(r, ...)` |
| `read.read_reproject_like(r, template)` | `await asyncread.read_reproject_like(r, template)` |
| `read.read_to_crs(r, dst_crs)` | `await asyncread.read_to_crs(r, dst_crs)` |
| `read.resize(r, resolution_dst=N)` | `await asyncread.resize(r, resolution_dst=N)` |
| `read.read_from_tile(r, x, y, z)` | `await asyncread.read_from_tile(r, x, y, z)` |

Crucially, the reprojection family **does not pre-load the entire raster**.
`asyncread.read_reproject` (and its wrappers `read_to_crs`,
`read_reproject_like`, `resize`, `read_from_tile`) compute the destination
grid, derive the matching input window, stream **only that window** over
the network, and warp it into the destination buffer with the same
`rasterio.warp.reproject` loop the sync path uses. The non-I/O code paths
(`_reproject_setup`, `_reproject_finalize`) are shared between
`read` and `asyncread` so the warp result is bit-identical.


In [11]:
# Pattern 1: asyncread.read_from_window — one await, one fetch.
from georeader import asyncread

gt = await asyncread.read_from_window(reader, win, trigger_load=True)
print(f"Type:  {type(gt).__name__}")
print(f"Shape: {gt.values.shape}")
# Without trigger_load you get the lazy view (an AsyncGeoData) — call
# `await view.load()` yourself to materialise.


Type:  GeoTensor
Shape: (3, 48, 64)


In [12]:
# Pattern 2: asyncread.read_from_bounds — geographic bounds, optional bounds-CRS.
import rasterio.warp

# Native-CRS bounds (UTM 31N for this fixture).
utm_bounds = (500080.0, 4599720.0, 500400.0, 4599960.0)
gt_native = await asyncread.read_from_bounds(reader, utm_bounds, trigger_load=True)
print(f"Native bounds:  shape={gt_native.values.shape}, crs={gt_native.crs}")

# Bounds in WGS84 — only the bounds are reprojected; the data stays in
# the reader's native CRS.
wgs_bounds = rasterio.warp.transform_bounds(reader.crs, "EPSG:4326", *utm_bounds)
gt_from_wgs = await asyncread.read_from_bounds(
    reader, wgs_bounds, crs_bounds="EPSG:4326", trigger_load=True
)
print(f"Bounds in WGS84: shape={gt_from_wgs.values.shape}, crs={gt_from_wgs.crs}")


Native bounds:  shape=(3, 24, 32), crs=EPSG:32631
Bounds in WGS84: shape=(3, 26, 33), crs=EPSG:32631


In [13]:
# Pattern 3: asyncread.read_to_crs — windowed reproject, no pre-load.
#
# Internally: compute the destination grid → derive the matching input
# polygon → `await asyncread.read_from_polygon(...)` for ONLY that window →
# rasterio.warp.reproject into the pre-allocated destination buffer. The
# entire raster never travels over the network just to warp one chip.

gt_wgs84 = await asyncread.read_to_crs(reader, dst_crs="EPSG:4326")
print(f"Reprojected to WGS84: shape={gt_wgs84.values.shape}, crs={gt_wgs84.crs}")

# read_reproject_like — match another GeoTensor's grid exactly. Also windowed.
target_grid = GeoTensor(
    values=np.zeros((3, 200, 200), dtype=np.int16),
    transform=from_origin(500000.0, 4600000.0, 12.0, 12.0),  # 12m pixels instead of 10m
    crs="EPSG:32631",
)
gt_aligned = await asyncread.read_reproject_like(reader, target_grid)
print(f"Aligned to template:  shape={gt_aligned.values.shape}, transform={gt_aligned.transform}")


Reprojected to WGS84: shape=(3, 218, 290), crs=EPSG:4326


Aligned to template:  shape=(3, 200, 200), transform=| 12.00, 0.00, 500000.00|
| 0.00,-12.00, 4600000.00|
| 0.00, 0.00, 1.00|


## Bandwidth-conscious tile reads via `asyncread.read_from_tile`

`asyncread.read_from_tile(reader, x, y, z)` fetches only the bytes for the
requested XYZ tile — it does **not** pre-load the whole raster. The
implementation composes the same pieces you would write by hand
(`mercantile.xy_bounds` → reproject bounds into the reader's native CRS →
windowed `read_from_polygon` → reproject the chip into Web Mercator), but
exposes them behind a single async call.

For a tile server fronting a large COG (e.g. Sentinel-2 at 10,980×10,980),
this is ~200× less bytes-over-the-wire per tile than the
"`await reader.load()` then sync `read.read_from_tile(gt, …)`" pattern.


In [14]:
# One-liner — internally fetches only the tile-region from the COG and
# reprojects the small chip to Web Mercator.
import mercantile

lon, lat = 3.0029, 41.5502
z = 14
tile = mercantile.tile(lon, lat, z)
tile_gt = await asyncread.read_from_tile(reader, x=tile.x, y=tile.y, z=tile.z)

print(f"Tile (z={z}, x={tile.x}, y={tile.y})")
print(f"  result CRS:   {tile_gt.crs}")
print(f"  result shape: {tile_gt.values.shape}")
print(f"  result bytes: {tile_gt.values.nbytes:,}")
print(f"  raster bytes: {reader.shape[0] * reader.shape[1] * reader.shape[2] * 2:,}  (whole COG)")


Tile (z=14, x=8328, y=6109)
  result CRS:   EPSG:3857
  result shape: (3, 256, 256)
  result bytes: 393,216
  raster bytes: 393,216  (whole COG)


## What this reader does NOT do

`async-geotiff` explicitly disclaims warp / resample / overview
auto-selection. `AsyncGeoTIFFReader` follows the same boundary by design;
the reproject/resize machinery lives in `georeader.asyncread`, which
streams only the input window required for the destination grid (the warp
loop itself is in-process CPU, not network I/O).

- **No automatic overview selection.** You pick the overview level
  explicitly via `overview_level=N` in the constructor; the reader won't
  guess based on a target resolution.
- **No band-select primitive.** `async-geotiff` reads all bands as a unit.
  Post-load slice the resulting `GeoTensor` if you only want some bands.
- **Not pickleable across processes.** The cached `_geotiff` handle
  doesn't survive a `multiprocessing` / `joblib` / Dask worker boundary.
  For multi-process use `RasterioReader`.
- **TIFF/COG only.** `async-geotiff` parses files as TIFF and rejects
  anything else. JP2 (the Sentinel-2 native L1C format), NetCDF, HDF5,
  GRIB all require `RasterioReader`. The JP2 case in particular is
  worth calling out — opening a `.jp2` raises
  `AsyncTiffException: unexpected magic bytes`.


## Tips and gotchas

- **Two-phase laziness.** Header on `open()`, pixels on `load()`. Properties
  raise `RuntimeError` before `open()`. Use `async with` if you want to skip
  the explicit `open` + `aclose` dance.
- **Sync view + async load.** `reader.read_from_window(w)` is sync and
  returns a windowed view (no I/O). `await view.load()` does the fetch.
  This is what makes the reader compose with the entire `read.*` module —
  see the compatibility section above.
- **Fan out via `gather`.** Building the views is instant; the concurrency
  lives in `asyncio.gather(*[v.load() for v in views])`.
- **Tile-align when you can.** Reading on tile boundaries
  (`reader.block_windows()`) avoids partial-tile fetches over the wire.
- **Pre-load for reproject.** `read.read_reproject` / `read_reproject_like` /
  `read_to_crs` need a `GeoTensor` (the in-memory short-circuit at
  `read.py:1605`). Call `gt = await reader.load()` once and pass `gt` in.
- **Not pickleable.** Open the reader fresh in each worker, or use
  `RasterioReader` for `multiprocessing`/Dask.
- **`store=` is required — no default.** Pick the right `obstore` store per
  cloud:
  - `obstore.store.S3Store(bucket=..., region=...)` for AWS S3
  - `obstore.store.GCSStore(bucket=..., ...)` for Google Cloud Storage
  - `obstore.store.AzureStore(container_name=..., ...)` for Azure Blob
  - `obstore.store.LocalStore(prefix=dir)` for local disk
  - `obstore.store.HTTPStore.from_url(url)` for HTTPS
- **Overviews are explicit.** `reader.overviews()` lists what's available;
  `reader.reader_overview(level)` (or `overview_level=N` at construction)
  pins a level. No auto-selection.
- **TIFF/COG only.** JP2, NetCDF, HDF5, GRIB → use `RasterioReader`.
- **Mask convention.** `async-geotiff`'s `RasterArray.mask` uses
  `True = valid` (inverse of numpy MA). The adapter handles this and
  substitutes `fill_value_default` where invalid.


## Going to the cloud

The flow is identical to the local fixture — swap `LocalStore` for the
appropriate `obstore.store.*` class. The reader doesn't care which cloud is
behind the store. The cell below runs against a real public bucket:
Element 84's `sentinel-cogs` mirror of Sentinel-2 L2A (anonymous access,
no credentials needed).

For credentialed buckets, look at the relevant `obstore.store.*`
constructor — each store accepts the standard cloud-specific auth:
env-vars, IAM roles, SAS tokens, etc. See the
[obstore docs](https://developmentseed.org/obstore/latest/) for the full
matrix.


In [15]:
from obstore.store import S3Store

# Anonymous access to the public Sentinel-2 L2A COG bucket.
cloud_store = S3Store(bucket="sentinel-cogs", region="us-west-2", skip_signature=True)
scene_band = "sentinel-s2-l2a-cogs/49/S/GV/2022/5/S2B_49SGV_20220527_0_L2A/B04.tif"

cloud_reader = await AsyncGeoTIFFReader.open(scene_band, store=cloud_store)
print(f"opened: shape={cloud_reader.shape}, crs={cloud_reader.crs}, res={cloud_reader.res}")

# Same fan-out as against the local fixture.
cloud_windows = [
    rasterio.windows.Window(col_off=5000 + (i % 4) * 256,
                            row_off=5000 + (i // 4) * 256,
                            width=256, height=256)
    for i in range(8)
]
chips = await asyncio.gather(
    *[cloud_reader.read_from_window(w).load() for w in cloud_windows]
)
print(f"{len(chips)} chips fetched concurrently; first: {chips[0].shape}, dtype={chips[0].values.dtype}")


opened: shape=(1, 10980, 10980), crs=EPSG:32649, res=(10.0, 10.0)


8 chips fetched concurrently; first: (1, 256, 256), dtype=uint16


## Cleanup

In [16]:
import shutil
shutil.rmtree(tmpdir)
